In [ ]:
!pip -q install pyomo
!apt-get -qq update
!apt-get -qq install -y glpk-utils


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import pandas as pd
import numpy as np
import math


In [ ]:
PATH = "/content/7B16E038 (1).xlsx"

In [ ]:
q = pd.read_excel(PATH, sheet_name="Quantities", header=None)

seg_rows = list(range(11, 17))

segments = [str(q.loc[r,1]).strip() for r in seg_rows]
demandA  = {str(q.loc[r,1]).strip(): float(q.loc[r,3]) for r in seg_rows}

segments_B = [str(q.loc[r,5]).strip() for r in seg_rows]
demandB    = {str(q.loc[r,5]).strip(): float(q.loc[r,7]) for r in seg_rows}

totalA = sum(demandA.values())
totalB = sum(demandB.values())
totalT = totalA + totalB

segments, totalA, totalB, totalT


(['Segment C1',
  'Segment D1a',
  'Segment D1b',
  'Segment C2a',
  'Segment C2b',
  'Segment D2'],
 2838600.0,
 363315.35,
 3201915.35)

In [ ]:
quotes = pd.read_excel(PATH, sheet_name="Quotes", header=None)

suppliers = [f"Supplier {i}" for i in range(1, 8)]
seg_rows_A_quarry = [10,11,12,13,14,15]
seg_rows_A_gravel = [19,20,21,22,23,24]
seg_rows_B        = [28,29,30,31,32,33]

def row_vals(r):
    vals = quotes.loc[r, 3:9].tolist()
    return dict(zip(suppliers, vals))

# supply costs
supply_A_quarry = row_vals(8)
supply_A_gravel = row_vals(17)
supply_B        = row_vals(26)

# hauling costs by segment
haul_A_quarry = {seg: row_vals(r) for seg, r in zip(segments, seg_rows_A_quarry)}
haul_A_gravel = {seg: row_vals(r) for seg, r in zip(segments, seg_rows_A_gravel)}
haul_B        = {seg: row_vals(r) for seg, r in zip(segments, seg_rows_B)}

sources_A = []
for s in suppliers:
    if not (isinstance(supply_A_quarry[s], float) and math.isnan(supply_A_quarry[s])):
        sources_A.append((s, "quarry"))
    if not (isinstance(supply_A_gravel[s], float) and math.isnan(supply_A_gravel[s])):
        sources_A.append((s, "gravel"))

# Delivered $/ton
costA = {}  # keyed by (supplier, mode, segment)
for s, mode in sources_A:
    for seg in segments:
        if mode == "quarry":
            costA[(s, mode, seg)] = float(supply_A_quarry[s] + haul_A_quarry[seg][s])
        else:
            costA[(s, mode, seg)] = float(supply_A_gravel[s] + haul_A_gravel[seg][s])

costB = {(s, seg): float(supply_B[s] + haul_B[seg][s]) for s in suppliers for seg in segments}

len(sources_A), sources_A[:5]


(10,
 [('Supplier 1', 'quarry'),
  ('Supplier 2', 'quarry'),
  ('Supplier 2', 'gravel'),
  ('Supplier 3', 'quarry'),
  ('Supplier 4', 'quarry')])

# Roadway Construction Procurement: Mathematical Model

---

## 1. Sets and Indices
* $S$: Set of **suppliers**, indexed by $s \in \{1, 2, 3, 4, 5, 6, 7\}$
* $J$: Set of **roadway segments**, indexed by $j \in \{C1, D1a, D1b, D1c, D1d, D2\}$
* $M$: Set of **material types** $\{A, B\}$ (A: Granular A, B: Granular B)
* $K$: Set of valid **Granular A sourcing options**, indexed by $(s, k)$, where $k \in \{\text{quarry, gravel}\}$

## 2. Parameters
* $D_j^A, D_j^B$: Demand for Granular A and B in segment $j$ (tons)
* $c_{s,k,j}^A$: Delivered cost per ton of Granular A (Supplier $s$, Mode $k$, Segment $j$)
* $c_{s,j}^B$: Delivered cost per ton of Granular B (Supplier $s$, Segment $j$)
* $\alpha = 0.50$: Maximum allowable supplier concentration ratio
* $T$: Total project demand $\left(\sum D_j^A + \sum D_j^B\right)$

---

## 3. Model 1: Base Model (No Discount)

### Decision Variables
* $x_{s,k,j}^A \ge 0$: Tons of Granular A supplied
* $x_{s,j}^B \ge 0$: Tons of Granular B supplied

### Objective Function
Minimize total procurement cost:
$$\min \sum_{j \in J} \sum_{(s,k) \in K} c_{s,k,j}^A x_{s,k,j}^A + \sum_{j \in J} \sum_{s \in S} c_{s,j}^B x_{s,j}^B$$

### Key Constraints
1. **Demand Satisfaction**: $\sum x_{s,k,j}^A = D_j^A$ and $\sum x_{s,j}^B = D_j^B$ for all $j$.
2. **Concentration**: Total volume per supplier $\le \alpha T$.
3. **Capacity**:
    * Supplier 4: $\le 1,000,000$ tons
    * Supplier 7: $\le 300,000$ tons
    * Supplier 2 (Gravel): $\le 300,000$ tons

---

## 4. Model 2: Enhanced Model (Supplier 3 Discount)
*Supplier 3 offers 3% off if total volume $Q > 600,000$ tons.*

### Linearization Logic
To handle the "if-then" discount without breaking the linear solver, we use a binary variable $y \in \{0,1\}$ and auxiliary variables $z_j^M$ to represent the product $x \cdot y$.

**Discount Trigger (Big-M):**
* $Q \ge 600,000 y$
* $Q \le 600,000 + M(1-y)$

**Linearization Constraints (for $M \in \{A, B\}$):**
* $z_j^M \le x_{3,j}^M$
* $z_j^M \le D_j^M y$
* $z_j^M \ge x_{3,j}^M - D_j^M(1-y)$

#3) Pyomo implementation
##3.1 base discount     

In [ ]:

import pyomo.environ as pyo

def build_base_model(segments, suppliers, sources_A, demandA, demandB, costA, costB,
                     cap_total=None, cap_A_gravel_supplier2=300_000, alpha=0.50):
    """
    cap_total: dict like {"Supplier 4": 1_000_000, "Supplier 7": 300_000}
    """
    if cap_total is None:
        cap_total = {}

    model = pyo.ConcreteModel()

    model.J = pyo.Set(initialize=segments)
    model.S = pyo.Set(initialize=suppliers)
    model.KA = pyo.Set(initialize=sources_A, dimen=2)   # (supplier, mode)

    # decision vars
    model.xA = pyo.Var(model.KA, model.J, domain=pyo.NonNegativeReals)
    model.xB = pyo.Var(model.S,  model.J, domain=pyo.NonNegativeReals)

    # objective
    def obj_rule(m):
        return sum(costA[(s,mode,j)] * m.xA[(s,mode), j] for (s,mode) in m.KA for j in m.J) + \
               sum(costB[(s,j)]       * m.xB[s, j]       for s in m.S for j in m.J)
    model.OBJ = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # demands
    def demA_rule(m, j):
        return sum(m.xA[(s,mode), j] for (s,mode) in m.KA) == demandA[j]
    model.DEM_A = pyo.Constraint(model.J, rule=demA_rule)

    def demB_rule(m, j):
        return sum(m.xB[s, j] for s in m.S) == demandB[j]
    model.DEM_B = pyo.Constraint(model.J, rule=demB_rule)

    # concentration limit
    T = sum(demandA.values()) + sum(demandB.values())
    def conc_rule(m, s):
        A_from_s = sum(m.xA[(ss,mode), j] for (ss,mode) in m.KA if ss == s for j in m.J)
        B_from_s = sum(m.xB[s, j] for j in m.J)
        return A_from_s + B_from_s <= alpha * T
    model.CONC = pyo.Constraint(model.S, rule=conc_rule)

    # total capacity
    def cap_rule(m, s):
        if s not in cap_total:
            return pyo.Constraint.Skip
        A_from_s = sum(m.xA[(ss,mode), j] for (ss,mode) in m.KA if ss == s for j in m.J)
        B_from_s = sum(m.xB[s, j] for j in m.J)
        return A_from_s + B_from_s <= cap_total[s]
    model.CAP = pyo.Constraint(model.S, rule=cap_rule)

    # Supplier 2
    def s2_gravel_rule(m):
        if ("Supplier 2", "gravel") not in sources_A:
            return pyo.Constraint.Skip
        return sum(m.xA[("Supplier 2","gravel"), j] for j in m.J) <= cap_A_gravel_supplier2
    model.S2_GRAVEL = pyo.Constraint(rule=s2_gravel_rule)

    return model



In [ ]:
cap_total = {"Supplier 4": 1_000_000, "Supplier 7": 300_000}

m_base = build_base_model(segments, suppliers, sources_A, demandA, demandB, costA, costB,
                          cap_total=cap_total, alpha=0.50)

solver = pyo.SolverFactory("glpk")
res = solver.solve(m_base, tee=False)
print(res.solver.termination_condition)
print("Base objective ($):", pyo.value(m_base.OBJ))


optimal
Base objective ($): 45172082.542212285


3.2 Enhanced model (Supplier 3 discount)

In [ ]:
def build_discount_model(segments, suppliers, sources_A, demandA, demandB, costA, costB,
                         cap_total=None, cap_A_gravel_supplier2=300_000, alpha=0.50,
                         disc_supplier="Supplier 3", threshold=600_000, disc_rate=0.03):
    if cap_total is None:
        cap_total = {}

    model = pyo.ConcreteModel()

    model.J  = pyo.Set(initialize=segments)
    model.S  = pyo.Set(initialize=suppliers)
    model.KA = pyo.Set(initialize=sources_A, dimen=2)

    model.xA = pyo.Var(model.KA, model.J, domain=pyo.NonNegativeReals)
    model.xB = pyo.Var(model.S,  model.J, domain=pyo.NonNegativeReals)

    # discount binary
    model.y = pyo.Var(domain=pyo.Binary)

    model.zA = pyo.Var(model.J, domain=pyo.NonNegativeReals)
    model.zB = pyo.Var(model.J, domain=pyo.NonNegativeReals)

    # Objective = base cost - discount on Miller
    def obj_rule(m):
        base = sum(costA[(s,mode,j)] * m.xA[(s,mode), j] for (s,mode) in m.KA for j in m.J) + \
               sum(costB[(s,j)]       * m.xB[s, j]       for s in m.S for j in m.J)

        # subtract discount if y=1 (via zA,zB)
        disc = disc_rate * (
            sum(costA[(disc_supplier,"quarry",j)] * m.zA[j] for j in m.J) +
            sum(costB[(disc_supplier,j)]          * m.zB[j] for j in m.J)
        )
        return base - disc

    model.OBJ = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # Demand constraints
    model.DEM_A = pyo.Constraint(model.J, rule=lambda m,j: sum(m.xA[(s,mode),j] for (s,mode) in m.KA) == demandA[j])
    model.DEM_B = pyo.Constraint(model.J, rule=lambda m,j: sum(m.xB[s,j] for s in m.S) == demandB[j])

    # concentration
    T = sum(demandA.values()) + sum(demandB.values())
    model.CONC = pyo.Constraint(model.S, rule=lambda m,s:
        (sum(m.xA[(ss,mode), j] for (ss,mode) in m.KA if ss==s for j in m.J) +
         sum(m.xB[s,j] for j in m.J)) <= alpha*T
    )

    # total capacity
    def cap_rule(m, s):
        if s not in cap_total:
            return pyo.Constraint.Skip
        A_from_s = sum(m.xA[(ss,mode), j] for (ss,mode) in m.KA if ss == s for j in m.J)
        B_from_s = sum(m.xB[s, j] for j in m.J)
        return A_from_s + B_from_s <= cap_total[s]
    model.CAP = pyo.Constraint(model.S, rule=cap_rule)

    # Supplier 2 gravel pit A-only cap
    model.S2_GRAVEL = pyo.Constraint(
        rule=lambda m: sum(m.xA[("Supplier 2","gravel"), j] for j in m.J) <= cap_A_gravel_supplier2
        if ("Supplier 2","gravel") in sources_A else pyo.Constraint.Skip
    )

    # Discount trigger
    def miller_total(m):
        A3 = sum(m.xA[(disc_supplier,"quarry"), j] for j in m.J) if (disc_supplier,"quarry") in sources_A else 0
        B3 = sum(m.xB[disc_supplier, j] for j in m.J)
        return A3 + B3

    M = T  # big-M
    model.TRIG_LO = pyo.Constraint(expr=miller_total(model) >= threshold * model.y)
    model.TRIG_HI = pyo.Constraint(expr=miller_total(model) <= threshold + M * model.y)

    # Linearization: zA[j] = xA3[j] * y, zB[j] = xB3[j] * y
    for j in segments:
        # bounds for big-M per segment (safe and tight)
        MA = demandA[j]
        MB = demandB[j]

        xA3j = model.xA[(disc_supplier,"quarry"), j]
        xB3j = model.xB[disc_supplier, j]

        model.add_component(f"zA_ub1_{j}", pyo.Constraint(expr=model.zA[j] <= xA3j))
        model.add_component(f"zA_ub2_{j}", pyo.Constraint(expr=model.zA[j] <= MA * model.y))
        model.add_component(f"zA_lb_{j}",  pyo.Constraint(expr=model.zA[j] >= xA3j - MA * (1-model.y)))

        model.add_component(f"zB_ub1_{j}", pyo.Constraint(expr=model.zB[j] <= xB3j))
        model.add_component(f"zB_ub2_{j}", pyo.Constraint(expr=model.zB[j] <= MB * model.y))
        model.add_component(f"zB_lb_{j}",  pyo.Constraint(expr=model.zB[j] >= xB3j - MB * (1-model.y)))

    return model


In [ ]:
m_disc = build_discount_model(segments, suppliers, sources_A, demandA, demandB, costA, costB,
                              cap_total=cap_total, alpha=0.50)

res2 = solver.solve(m_disc, tee=False)
print(res2.solver.termination_condition)
print("Discount objective ($):", pyo.value(m_disc.OBJ))
print("Discount active? y =", pyo.value(m_disc.y))


optimal
Discount objective ($): 44757308.05283728
Discount active? y = 1.0


In [ ]:
def summarize_solution(model, segments, suppliers, sources_A, label=""):
    rows = []

    # A flows
    for (s,mode) in sources_A:
        qty = sum(pyo.value(model.xA[(s,mode), j]) for j in segments)
        if qty > 1e-6:
            rows.append({"Material":"A", "Supplier":s, "Mode":mode, "Qty_ton":qty})

    # B flows
    for s in suppliers:
        qty = sum(pyo.value(model.xB[s, j]) for j in segments)
        if qty > 1e-6:
            rows.append({"Material":"B", "Supplier":s, "Mode":"quarry", "Qty_ton":qty})

    df = pd.DataFrame(rows).sort_values(["Material","Supplier","Mode"])
    print(label, "Total cost:", pyo.value(model.OBJ))
    return df

df_base = summarize_solution(m_base, segments, suppliers, sources_A, label="BASE")
df_disc = summarize_solution(m_disc, segments, suppliers, sources_A, label="DISCOUNT")

df_base, df_disc


BASE Total cost: 45172082.542212285
DISCOUNT Total cost: 44757308.05283728


(  Material    Supplier    Mode     Qty_ton
 1        A  Supplier 2  gravel   300000.00
 0        A  Supplier 2  quarry   663537.75
 2        A  Supplier 3  quarry   872291.25
 3        A  Supplier 4  gravel  1000000.00
 4        A  Supplier 7  quarry     2771.00
 5        B  Supplier 1  quarry   213591.55
 6        B  Supplier 2  quarry   149723.80,
   Material    Supplier    Mode     Qty_ton
 1        A  Supplier 2  gravel   300000.00
 0        A  Supplier 2  quarry   663537.75
 2        A  Supplier 3  quarry   872291.25
 3        A  Supplier 4  gravel  1000000.00
 4        A  Supplier 7  quarry     2771.00
 5        B  Supplier 1  quarry   213591.55
 6        B  Supplier 2  quarry   149723.80)

In [ ]:
from IPython.display import display

def segment_breakdown_A(model):
    out = []
    for j in segments:
        for (s, mode) in sources_A:
            v = pyo.value(model.xA[(s,mode), j])
            if v is not None and v > 1e-6:
                out.append({"Segment": j, "Supplier": s, "Mode": mode, "QtyA": v})
    return pd.DataFrame(out)

def segment_breakdown_B(model):
    out = []
    for j in segments:
        for s in suppliers:
            v = pyo.value(model.xB[s, j])
            if v is not None and v > 1e-6:
                out.append({"Segment": j, "Supplier": s, "QtyB": v})
    return pd.DataFrame(out)

A_base = segment_breakdown_A(m_base)
B_base = segment_breakdown_B(m_base)
A_disc = segment_breakdown_A(m_disc)
B_disc = segment_breakdown_B(m_disc)

display(A_base.head(20))
display(B_base.head(20))
display(A_disc.head(20))
display(B_disc.head(20))


,Segment,Supplier,Mode,QtyA
0,Segment C1,Supplier 3,quarry,497940.750000
1,Segment D1a,Supplier 3,quarry,374350.500000
2,Segment D1b,Supplier 4,gravel,517903.750000
3,Segment D1b,Supplier 7,quarry,2771.000000
4,Segment C2a,Supplier 2,quarry,456533.178356
5,Segment C2b,Supplier 2,quarry,207004.571644
6,Segment C2b,Supplier 2,gravel,300000.000000
7,Segment D2,Supplier 4,gravel,482096.250000


,Segment,Supplier,QtyB
0,Segment C1,Supplier 1,76739.70000
1,Segment D1a,Supplier 2,48250.85000
2,Segment D1b,Supplier 2,43255.00000
3,Segment C2a,Supplier 1,71054.51825
4,Segment C2b,Supplier 1,65797.33175
5,Segment D2,Supplier 2,58217.95000


,Segment,Supplier,Mode,QtyA
0,Segment C1,Supplier 3,quarry,497940.750000
1,Segment D1a,Supplier 3,quarry,374350.500000
2,Segment D1b,Supplier 4,gravel,520674.750000
3,Segment C2a,Supplier 2,quarry,456533.178356
4,Segment C2b,Supplier 2,quarry,207004.571644
5,Segment C2b,Supplier 2,gravel,300000.000000
6,Segment D2,Supplier 4,gravel,479325.250000
7,Segment D2,Supplier 7,quarry,2771.000000


,Segment,Supplier,QtyB
0,Segment C1,Supplier 1,76739.70000
1,Segment D1a,Supplier 2,48250.85000
2,Segment D1b,Supplier 2,43255.00000
3,Segment C2a,Supplier 1,71054.51825
4,Segment C2b,Supplier 1,65797.33175
5,Segment D2,Supplier 2,58217.95000
